In [1]:
import os
import time
import torch
from threading import Thread
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, TextIteratorStreamer
from qwen_vl_utils import process_vision_info

os.environ["HF_HOME"] = "/home/nielly/hf_datasets_cache"

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

print("Loading model...")
t0 = time.time()

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="sdpa", 
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Done in {time.time()-t0:.1f}s")
print(f"Device map: {model.hf_device_map}")

Loading model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Done in 23.6s
Device map: {'model.visual': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_model.layers.24': 1,

In [ ]:
def run_streaming(messages, max_new_tokens=512):
    # Build prompt + process vision
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    images, videos, video_kwargs = process_vision_info(
        messages, return_video_kwargs=True
    )

    if videos is not None:
        videos, video_metadatas = zip(*videos)
        videos, video_metadatas = list(videos), list(video_metadatas)
    else:
        video_metadatas = None

    if video_kwargs:
        video_kwargs = {
            k: v[0] if isinstance(v, list) and len(v) == 1 else v
            for k, v in video_kwargs.items()
        }
    
    inputs = processor(
        text=[text],
        images=images,
        videos=videos,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        #max_new_tokens=max_new_tokens,
        do_sample=False,
        **video_kwargs,
    ).to(model.device)

    print(f"Input tokens: {inputs['input_ids'].shape[-1]:,}")
    print("─" * 50)

    t0 = time.time()
    generated_ids = model.generate(**inputs)
    print(generated_ids)

    print(f"\n{'─'*50}")
    print(f"Generated {len(generated_ids)} words in {time.time()-t0:.1f}s")

    return full_response

In [11]:
VIDEO_PATH = "/scratch/izar/nielly/tsu_dataset/Videos_mp4/P02T01C06.mp4"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": VIDEO_PATH,
                "nframes": 32,   # hard cap on frame count
                # "fps": 0.5,    # alternative: use fps instead
            },
            {
                "type": "text",
                "text": "Describe what happens in this video in chronological order."
            },
        ],
    }
]

response = run_streaming(messages)

[transformers] Qwen3VL requires frame timestamps to construct prompts, but the `fps` of the input video could not be inferred. Probably `video_metadata` was missing from inputs and you passed pre-sampled frames. Defaulting to `fps=24`. Please provide `video_metadata` for more accurate results.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: 4,946
──────────────────────────────────────────────────
A woman is standing in a modern kitchen, preparing a drink. She is wearing a patterned shirt and has short hair. She is standing in front of a stainless steel stove and is using a coffee maker. She pours coffee into a cup and then adds sugar. She then walks over to the sink and washes her hands. She then walks back to the counter and pours the coffee into a mug. She then walks over to the living room and sits down on the couch. She then picks up the mug and takes a sip of the coffee. She then puts the mug down and watches television.
──────────────────────────────────────────────────
Generated 108 words in 24.0s


In [23]:
from ipywidgets import Video

Video.from_file(VIDEO_PATH)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

In [ ]:
from decord import VideoReader
import numpy as np

vr = VideoReader(VIDEO_PATH)
total_frames = len(vr)
indices = np.linspace(0, total_frames - 1, 32).astype(int)
frames = vr.get_batch(indices).asnumpy()
print(f"Frame shape fed to model: {frames[0].shape}")  # (H, W, C)
print(f"Tokens per frame: ~{(frames[0].shape[0]//28) * (frames[0].shape[1]//28)}")